# 04 - Zillow Data Pipeline

Loads Zillow ZHVI county-level data, assigns Karl & Koss climate regions, and computes
region-minus-affected baselines for each county-month storm observation.

**Inputs:**
- Zillow ZHVI county CSV (fetched directly)
- `../data/processed/storm_events.pkl` — to identify affected counties per month

**Output:** `../data/processed/zillow_panel.pkl` — one row per county-month storm observation
with ZHVI indexed to 100 at storm month and regional baseline ZHVI for comparison

**Baseline definition:** Equal-weighted mean ZHVI of all counties in the same Karl & Koss
climate region that had no recorded storm event in that month.

**Karl & Koss climate regions (1984):**
Nine climatologically-defined regions used throughout NOAA climate research.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

## Karl & Koss Climate Region Lookup
State-level assignment per Karl, T.R. and Koss, W.J. (1984).

In [2]:
# Keyed by 2-digit state FIPS string
KARL_KOSS_REGIONS = {
    # Northeast: CT, DE, ME, MD, MA, NH, NJ, NY, PA, RI, VT
    '09': 'Northeast', '10': 'Northeast', '23': 'Northeast', '24': 'Northeast',
    '25': 'Northeast', '33': 'Northeast', '34': 'Northeast', '36': 'Northeast',
    '42': 'Northeast', '44': 'Northeast', '50': 'Northeast',
    # East North Central: IA, MI, MN, WI  (note: Karl & Koss excludes IL, IN, OH)
    '19': 'East North Central', '26': 'East North Central',
    '27': 'East North Central', '55': 'East North Central',
    # Central: IL, IN, KY, MO, OH, TN, WV
    '17': 'Central', '18': 'Central', '21': 'Central', '29': 'Central',
    '39': 'Central', '47': 'Central', '54': 'Central',
    # Southeast: AL, FL, GA, NC, SC, VA
    '01': 'Southeast', '12': 'Southeast', '13': 'Southeast',
    '37': 'Southeast', '45': 'Southeast', '51': 'Southeast',
    # South: AR, LA, MS, OK, TX
    '05': 'South', '22': 'South', '28': 'South', '40': 'South', '48': 'South',
    # West North Central: KS, MN, MO, NE, ND, SD  (MN/MO split across regions in K&K)
    '20': 'West North Central', '31': 'West North Central',
    '38': 'West North Central', '46': 'West North Central',
    # Southwest: AZ, CO, NM, UT
    '04': 'Southwest', '08': 'Southwest', '35': 'Southwest', '49': 'Southwest',
    # Northwest: ID, MT, OR, WA, WY
    '16': 'Northwest', '30': 'Northwest', '41': 'Northwest',
    '53': 'Northwest', '56': 'Northwest',
    # West: AK, CA, HI, NV
    '02': 'West', '06': 'West', '15': 'West', '32': 'West',
    # DC
    '11': 'Northeast',
}

## ZillowDataParser
Unchanged from `NOAA_Storm_and_Zillow_Data_Pipeline.ipynb`

In [3]:
class ZillowDataParser():
    DTYPES = {
        'RegionID': 'Int64',
        'SizeRank': 'Int64',
        'RegionName': 'object',
        'RegionType': 'object',
        'StateName': 'object',
        'State': 'object',
        'Metro': 'object',
        'StateCodeFIPS': 'Int64',
        'MunicipalCodeFIPS': 'Int64'
    }
    
    META_DATA_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
                      'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS',
                      'latitude', 'longitude']
    
    def __init__(self, regional_url, national_url):
        self.download_data(regional_url, national_url)
        
    def download_data(self, regional_url, national_url):
        regional_data_raw = pd.read_csv(regional_url)
        national_data_raw = pd.read_csv(national_url)
        national = national_data_raw.loc[national_data_raw["RegionName"] == "United States"]
        self.df = pd.concat([regional_data_raw, national], sort=False, ignore_index=True)
        self.df = self.df.astype(self.DTYPES)
        all_columns = self.df.columns.tolist() + [col for col in self.META_DATA_COLS if col not in self.df.columns]
        self.df = self.df.reindex(columns=all_columns)

    def date_cols(self):
        dates = [col for col in self.df.columns if col not in self.META_DATA_COLS]
        dates = sorted(dates, key=pd.to_datetime)
        return list(dates)

    def get_monthly_panel(self):
        date_cols = self.date_cols()
        long_df = self.df.melt(
            id_vars=self.META_DATA_COLS,
            value_vars=date_cols,
            var_name='date',
            value_name='zhvi'
        )
        long_df['date'] = pd.to_datetime(long_df['date'])
        return long_df

## Load Zillow Data

In [4]:
zillow_county_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
zillow_msa_url    = "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"

# Save raw files for reproducibility
import urllib.request
county_raw_path = Path('../data/raw/zillow_county_zhvi.csv')
msa_raw_path    = Path('../data/raw/zillow_msa_zhvi.csv')

if not county_raw_path.exists():
    print('Downloading Zillow county ZHVI...')
    urllib.request.urlretrieve(zillow_county_url, county_raw_path)
else:
    print('Using cached zillow_county_zhvi.csv')

if not msa_raw_path.exists():
    print('Downloading Zillow MSA ZHVI...')
    urllib.request.urlretrieve(zillow_msa_url, msa_raw_path)
else:
    print('Using cached zillow_msa_zhvi.csv')

zillow = ZillowDataParser(str(county_raw_path), str(msa_raw_path))
print(f'Loaded {len(zillow.df):,} rows, {len(zillow.date_cols())} months')

Using cached zillow_county_zhvi.csv
Using cached zillow_msa_zhvi.csv
Loaded 3,074 rows, 314 months


## Build Long Panel and Assign Climate Regions

In [5]:
panel = zillow.get_monthly_panel()

# Keep only county-level rows (drop national, MSA etc)
panel = panel[panel['RegionType'] == 'county'].copy()

# Build stcofips
panel['state_fips']  = panel['StateCodeFIPS'].astype(str).str.zfill(2)
panel['county_fips'] = panel['MunicipalCodeFIPS'].astype(str).str.zfill(3)
panel['stcofips']    = panel['state_fips'] + panel['county_fips']

# Assign Karl & Koss climate region
panel['climate_region'] = panel['state_fips'].map(KARL_KOSS_REGIONS)

# Extract year and month
panel['year']  = panel['date'].dt.year
panel['month'] = panel['date'].dt.month

# Filter to storm event years only
panel = panel[panel['year'].between(2020, 2025)].copy()

# Drop rows with no ZHVI or no region assignment
n_before = len(panel)
panel = panel.dropna(subset=['zhvi', 'climate_region'])
print(f'Dropped {n_before - len(panel):,} rows with missing ZHVI or unassigned region')
print(f'Panel shape: {panel.shape}')
print(f'Climate regions: {sorted(panel["climate_region"].unique())}')

Dropped 1,695 rows with missing ZHVI or unassigned region
Panel shape: (219561, 19)
Climate regions: ['Central', 'East North Central', 'Northeast', 'Northwest', 'South', 'Southeast', 'Southwest', 'West', 'West North Central']


## Load Storm Events and Compute Region-Minus-Affected Baseline

In [6]:
storms = pd.read_pickle('../data/processed/storm_events.pkl')

# Set of affected county-month pairs
affected = set(zip(storms['stcofips'], storms['year'], storms['month']))
print(f'Affected county-month observations: {len(affected):,}')

# Flag affected counties in the full panel
panel['affected'] = panel.apply(
    lambda r: (r['stcofips']b, r['year'], r['month']) in affected, axis=1
)

print(f'Affected rows in Zillow panel: {panel["affected"].sum():,}')
print(f'Unaffected rows (baseline pool): {(~panel["affected"]).sum():,}')

Affected county-month observations: 3,892
Affected rows in Zillow panel: 3,833
Unaffected rows (baseline pool): 215,728


In [76]:
# Compute regional baseline: equal-weighted mean ZHVI of unaffected counties
# per climate_region x year x month
# Primary baseline: unaffected counties only
baseline_group_col = 'state_fips'

baseline_unaffected = (
    panel[~panel['affected']]
    .groupby([baseline_group_col, 'year', 'month'])['zhvi']
    .agg(baseline_zhvi='mean', baseline_n_counties='count')
    .reset_index()
)

# Fallback baseline: full region mean for months where no unaffected counties exist
baseline_full = (
    panel
    .groupby([baseline_group_col, 'year', 'month'])['zhvi']
    .agg(baseline_zhvi='mean', baseline_n_counties='count')
    .reset_index()
    .rename(columns={'baseline_zhvi': 'baseline_zhvi_full', 
                     'baseline_n_counties': 'baseline_n_counties_full'})
)

# Merge and fill gaps
baseline = baseline_unaffected.merge(baseline_full, on=[baseline_group_col, 'year', 'month'], how='right')
baseline['baseline_zhvi']      = baseline['baseline_zhvi'].fillna(baseline['baseline_zhvi_full'])
baseline['baseline_n_counties'] = baseline['baseline_n_counties'].fillna(0)
baseline = baseline.drop(columns=['baseline_zhvi_full', 'baseline_n_counties_full'])
baseline.head()

,state_fips,year,month,baseline_zhvi,baseline_n_counties
0,01,2020,1,138284.664081,62.0
1,01,2020,2,136554.852929,61.0
2,01,2020,3,140551.429671,60.0
3,01,2020,4,142501.877033,43.0
4,01,2020,5,142238.132599,66.0


In [8]:
# Keep only storm-affected county-months for the analysis dataset
affected_panel = panel[panel['affected']].copy()

# Join regional baseline
affected_panel = affected_panel.merge(
    baseline,
    on=[baseline_group_col, 'year', 'month'],
    how='left'
)

# Compute ZHVI deviation from baseline (positive = above baseline)
affected_panel['zhvi_vs_baseline'] = affected_panel['zhvi'] - affected_panel['baseline_zhvi']

print(f'Affected panel shape: {affected_panel.shape}')
print(f'Rows missing baseline: {affected_panel["baseline_zhvi"].isnull().sum()}')
affected_panel[['stcofips', 'year', 'month',baseline_group_col,'climate_region', 'zhvi', 'baseline_zhvi', 'zhvi_vs_baseline', 'baseline_n_counties']].head(10)

Affected panel shape: (3833, 23)
Rows missing baseline: 0


,stcofips,year,month,state_fips,climate_region,zhvi,baseline_zhvi,zhvi_vs_baseline,baseline_n_counties
0,48113,2020,1,48,South,222706.233255,181656.585906,41049.647349,205.0
1,13067,2020,1,13,Southeast,283335.805032,150808.180018,132527.625014,151.0
2,12127,2020,1,12,Southeast,217682.349775,208014.367043,9667.982733,64.0
3,12069,2020,1,12,Southeast,246273.864943,208014.367043,38259.497900,64.0
4,45051,2020,1,45,Southeast,208951.876592,169953.459180,38998.417412,41.0
5,29077,2020,1,29,Central,155137.280019,139150.414639,15986.865380,106.0
6,48309,2020,1,48,South,169822.101315,181656.585906,-11834.484590,205.0
7,45007,2020,1,45,Southeast,169026.424567,169953.459180,-927.034612,41.0
8,28033,2020,1,28,South,208742.914377,127422.876665,81320.037712,67.0
9,48251,2020,1,48,South,235481.295249,181656.585906,53824.709344,205.0


## Load Storm Events and Compute Neighbor-Unaffected Baseline

In [95]:
# Adjacent Counties Baseline
# Compute baseline from local unaffected counties
adj_df = pd.read_csv('../data/raw/county_adjacency.csv')
adj_df['County GEOID'] = adj_df['County GEOID'].astype(int).astype(str).str.zfill(5)
adj_df['Neighbor GEOID'] = adj_df['Neighbor GEOID'].astype(int).astype(str).str.zfill(5)

def get_local_baseline(target_fips):
    adj_df[adj_df['County GEOID'] == target_fips]['Neighbor GEOID'].values

# Create a dictionary where key = County, value = list of Neighbors
# We exclude the county itself if it appears in the neighbors list
neighbors = adj_df[adj_df['Neighbor GEOID'] != adj_df['County GEOID']].groupby('County GEOID')['Neighbor GEOID'].apply(list).to_dict()

neighbors = pd.DataFrame([
    (target, neighbor) for target, neighbor_list in neighbors.items()
        for neighbor in neighbor_list
], columns = ['target_fips', 'neighbor_fips'])
neighbors

,target_fips,neighbor_fips
0,01001,01021
1,01001,01047
2,01001,01051
3,01001,01085
4,01001,01101
...,...,...
18957,72153,72081
18958,72153,72093
18959,72153,72121
18960,78020,78030


In [113]:
# WINDOW Size is -3 and +9

panel['month_t'] = (panel['year']-2020)*12 + panel['month']

panel.sort_values(['stcofips', 'month_t'])

rolling_affected = (panel[['stcofips', 'month_t', 'affected']]
 .groupby('stcofips')['affected']
 .rolling(window=13, center=False).sum().shift(-9)
 .reset_index(level=0, drop=True)
)

panel['window_affected_count'] = rolling_affected
panel['is_clean_control'] = panel['window_affected_count'] == 0


,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,latitude,...,state_fips,county_fips,stcofips,climate_region,year,month,affected,month_t,window_affected_count,is_clean_control
737760,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,NaN,...,06,037,06037,West,2020,1,False,1,NaN,False
737761,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,NaN,...,17,031,17031,Central,2020,1,False,1,NaN,False
737762,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,NaN,...,48,201,48201,South,2020,1,False,1,NaN,False
737763,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,NaN,...,04,013,04013,Southwest,2020,1,False,1,NaN,False
737764,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,NaN,...,06,073,06073,West,2020,1,False,1,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
959082,846,3206,Banner County,county,NE,NE,"Scottsbluff, NE",31,7,NaN,...,31,007,31007,West North Central,2025,12,False,72,NaN,False
959083,1648,3207,Daggett County,county,UT,UT,NaN,49,9,NaN,...,49,009,49009,Southwest,2025,12,False,72,NaN,False
959084,1432,3208,Thomas County,county,NE,NE,NaN,31,171,NaN,...,31,171,31171,West North Central,2025,12,False,72,NaN,False
959085,2794,3212,McPherson County,county,NE,NE,"North Platte, NE",31,117,NaN,...,31,117,31117,West North Central,2025,12,False,72,NaN,False


In [115]:
# Adjacent Counties Baseline

# 2. Join the neighbors to their actual price data for each month
neighbors_df = neighbors.merge(
    panel[['stcofips','zhvi', 'year', 'month', 'month_t', 'is_clean_control']],
    left_on='neighbor_fips',
    right_on='stcofips'
)
neighbors_df = neighbors_df[neighbors_df['is_clean_control'] == True]
# 3. Calculate the average deviation of the neighbors for each target county, each month
neighbors_df = (neighbors_df
 .groupby(['target_fips', 'year', 'month'])
 .agg(baseline_zhvi=('zhvi','mean'), baseline_n_counties=('zhvi','count'))
 .reset_index()
)

neighbors_df


,target_fips,year,month,baseline_zhvi,baseline_n_counties
0,01001,2020,4,156740.639092,3
1,01001,2020,5,157654.839736,3
2,01001,2020,6,159154.988554,2
3,01001,2020,7,158208.445101,2
4,01001,2020,8,129614.499025,1
...,...,...,...,...,...
186086,56045,2024,11,400047.046535,4
186087,56045,2024,12,401442.785983,4
186088,56045,2025,1,402856.738117,4
186089,56045,2025,2,404465.915190,4


In [119]:
# Adjacent Counties Baseline
# Keep only storm-affected county-months for the analysis dataset
affected_panel = panel[panel['affected']].copy()

# Join neighbors baseline
affected_panel = affected_panel.merge(
    neighbors_df,
    left_on=['stcofips', 'year', 'month'],
    right_on=['target_fips', 'year', 'month'],
    how='left'
)

# Compute ZHVI deviation from baseline (positive = above baseline)
affected_panel['zhvi_vs_baseline'] = affected_panel['zhvi'] - affected_panel['baseline_zhvi']

affected_panel = affected_panel[affected_panel['baseline_n_counties'] > 1 ].copy()

print(f'Affected panel shape: {affected_panel.shape}')
print(f'Rows missing baseline: {affected_panel["baseline_zhvi"].isnull().sum()}')
affected_panel[['stcofips', 'year', 'month','climate_region', 'zhvi', 'baseline_zhvi', 'zhvi_vs_baseline', 'baseline_n_counties']].head(10)

Affected panel shape: (2662, 27)
Rows missing baseline: 0


,stcofips,year,month,climate_region,zhvi,baseline_zhvi,zhvi_vs_baseline,baseline_n_counties
195,12099,2020,4,Southeast,307151.680962,211047.834088,96103.846874,5.0
196,42003,2020,4,Northeast,172653.029658,218166.770754,-45513.741096,2.0
197,13121,2020,4,Southeast,315913.266226,275235.789839,40677.476387,4.0
198,12031,2020,4,Southeast,205632.104567,251265.924270,-45633.819703,3.0
199,39061,2020,4,Central,175189.427094,219458.175628,-44268.748534,5.0
201,34029,2020,4,Northeast,305009.359779,233357.466132,71651.893647,2.0
202,39153,2020,4,Central,144154.846592,177253.204154,-33098.357561,4.0
203,45045,2020,4,Southeast,221059.252310,247649.157278,-26589.904967,3.0
205,45079,2020,4,Southeast,158856.931893,176840.473897,-17983.542004,3.0
208,39151,2020,4,Central,136781.566365,155520.672861,-18739.106496,5.0


## Validate

In [120]:
assert affected_panel.duplicated(['stcofips', 'year', 'month']).sum() == 0, 'Duplicate county-month rows'
assert affected_panel['baseline_zhvi'].isnull().sum() == 0, 'Missing baseline — check region assignment'
assert affected_panel['stcofips'].str.len().eq(5).all(), 'FIPS not all 5 digits'
# assert affected_panel['baseline_n_counties'].gt(0).all(), 'Empty baseline'
affected_panel = affected_panel[~affected_panel['baseline_n_counties'].eq(0)]

print('All assertions passed')
print(f'\nBaseline county counts by region (min across all months):')
print(
    affected_panel.groupby(baseline_group_col)['baseline_n_counties']
    .min()
    .sort_values()
    .reset_index()
)

All assertions passed

Baseline county counts by region (min across all months):
   state_fips  baseline_n_counties
0          01                  2.0
1          29                  2.0
2          55                  2.0
3          31                  2.0
4          34                  2.0
5          35                  2.0
6          36                  2.0
7          37                  2.0
8          38                  2.0
9          39                  2.0
10         40                  2.0
11         42                  2.0
12         44                  2.0
13         45                  2.0
14         46                  2.0
15         47                  2.0
16         48                  2.0
17         51                  2.0
18         28                  2.0
19         27                  2.0
20         30                  2.0
21         25                  2.0
22         24                  2.0
23         22                  2.0
24         05                  2.0
25       

## Export

In [121]:
output_cols = [
    'stcofips', 'state_fips', 'county_fips', 'climate_region',
    'year', 'month',
    'zhvi', 'baseline_zhvi', 'zhvi_vs_baseline', 'baseline_n_counties'
]

out = affected_panel[output_cols].sort_values(['stcofips', 'year', 'month']).reset_index(drop=True)

out.to_pickle('../data/processed/zillow_panel.pkl')

print(f'Exported zillow_panel.pkl')
print(f'Shape: {out.shape}')
print(f'Columns: {out.columns.tolist()}')

Exported zillow_panel.pkl
Shape: (2662, 10)
Columns: ['stcofips', 'state_fips', 'county_fips', 'climate_region', 'year', 'month', 'zhvi', 'baseline_zhvi', 'zhvi_vs_baseline', 'baseline_n_counties']


In [122]:
baseline.to_pickle('../data/processed/baseline_lookup.pkl')
print(f'Exported baseline_lookup.pkl')

Exported baseline_lookup.pkl
